In [1]:
import time
import numpy as np

In [2]:
import dask
import dask.array

In [3]:
import h5py
import os
from datetime import datetime

In [4]:
from scipy import ndimage

In [5]:
sample = '024_large_02_3uls'
scan = '0004'
toppath = '/data/visitor/ihmi1620/id19/20261006'

# input paths
procpath = os.path.join(toppath, 'PROCESSED_DATA')
samplepath = os.path.join(procpath, sample, sample+'_'+scan)
int16_volumepath = os.path.join(samplepath, 'reconstructed_volumes','cast_volume')

In [6]:
def load_im_dask(datapath, name, ts=None):
    splitname = name.split('_')
    ts = int(splitname[-3][:-3])
    entry = '/entry'+f'{ts:04d}'+'/reconstruction/results/data'
    tsentry =  '/entry'+f'{ts:04d}'+'/reconstruction/configuration/acquisition/end_time'
    # print(name, entry, tsentry)
    path = os.path.join(datapath, name)
    file = h5py.File(path)
    da = dask.array.from_array(file[entry])
    byte_data = file[tsentry][()]
    timestamp = datetime.fromisoformat(byte_data.decode()).timestamp() # ChatGPT
    return da, timestamp

In [7]:
def stack_4D(datapath):
    names = os.listdir(datapath)
    names.sort()
    data = []
    timestamps = []
    for name in names:
        da, timestamp = load_im_dask(datapath, name)
        data.append(da)
        timestamps.append(timestamp)
    timestamps = np.array(timestamps)
    Stack = dask.array.stack(data, axis=3)
    return Stack, names, timestamps

In [8]:
Stack, names, timestamps = stack_4D(int16_volumepath)

In [9]:
im = Stack[:,:,:,30]

In [21]:
im = im.compute()

In [12]:
im.shape

(3008, 1024, 1024)

In [22]:
subim = im[1000:1600,200:600,200:600]

In [25]:
t0 = time.time()
grad = ndimage.gaussian_filter(subim, 1)
grad = np.gradient(grad, axis = 2)
print(time.time()-t0)

1.2445666790008545


In [24]:
t0 = time.time()
grad = ndimage.gaussian_filter(subim, 1, order=(0, 0, 1), mode="nearest")
print(time.time()-t0)

1.0748894214630127


In [11]:
subim

dask.array<getitem, shape=(600, 400, 400), dtype=uint16, chunksize=(382, 206, 206), chunktype=numpy.ndarray>

In [23]:
volume = dask.array.from_array(subim)
# volume = subim

In [24]:
volume

dask.array<array, shape=(600, 400, 400), dtype=uint16, chunksize=(419, 400, 400), chunktype=numpy.ndarray>

In [17]:
sigma = 1
rho = 3
truncate = 4

In [15]:
import dask.array as lib
import dask_image.ndfilters

In [28]:
    Vx = dask_image.ndfilters.gaussian_filter(volume, sigma, order=(0, 0, 1), mode="nearest", truncate=truncate)  # type: ignore
    Vy = dask_image.ndfilters.gaussian_filter(volume, sigma, order=(0, 1, 0), mode="nearest", truncate=truncate)  # type: ignore
    Vz = dask_image.ndfilters.gaussian_filter(volume, sigma, order=(1, 0, 0), mode="nearest", truncate=truncate)  # type: ignore

In [36]:
    S = lib.empty((6,) + volume.shape, dtype=volume.dtype)

    # Integrating elements of structure tensor (scipy uses sequence of 1D).
    lib.multiply(Vx, Vx, out=S[0])
    S[0] = dask_image.ndfilters.gaussian_filter(S[0], rho, mode="nearest",truncate=truncate)
    lib.multiply(Vy, Vy, out=S[1])
    S[1] = dask_image.ndfilters.gaussian_filter(S[1], rho, mode="nearest", truncate=truncate)
    lib.multiply(Vz, Vz, out=S[2])
    S[2] = dask_image.ndfilters.gaussian_filter(S[2], rho, mode="nearest", truncate=truncate)
    lib.multiply(Vx, Vy, out=S[3])
    S[3] = dask_image.ndfilters.gaussian_filter(S[3], rho, mode="nearest", truncate=truncate)
    lib.multiply(Vx, Vz, out=S[4])
    S[4] = dask_image.ndfilters.gaussian_filter(S[4], rho, mode="nearest", truncate=truncate)
    lib.multiply(Vy, Vz, out=S[5])
    S[5] = dask_image.ndfilters.gaussian_filter(S[5], rho, mode="nearest", truncate=truncate)

In [39]:
S

dask.array<setitem, shape=(6, 600, 400, 400), dtype=uint16, chunksize=(6, 223, 223, 223), chunktype=numpy.ndarray>

In [40]:
input_shape = S.shape
S = S.reshape(6, -1)   
# Create v vector.
v = lib.array([[2 * lib.pi / 3], [4 * lib.pi / 3]], dtype=S.dtype)

In [42]:
val = lib.empty((3,) + S.shape[1:], dtype=S.dtype)
vec = lib.empty((3,) + S.shape[1:], dtype=S.dtype)
tmp = lib.empty((4,) + S.shape[1:], dtype=S.dtype)
B03 = val
B36 = vec

In [44]:
    # Views for B.
    B0 = B03[0]
    B1 = B03[1]
    B2 = B03[2]
    B3 = B36[0]
    B4 = B36[1]
    B5 = B36[2]


In [45]:
    q = lib.add(S[0], S[1], out=tmp[0])
    q += S[2]
    q /= 3


In [48]:
Sq = lib.subtract(S[:3], q, out=B03)

In [54]:
Snp = S.compute()

In [51]:
s = lib.einsum("ij,ij->j", S[3:], S[3:], out=tmp[1])
s *= 2

In [58]:
s = np.einsum("ij,ij->j", Snp[3:], Snp[3:])
s *= 2

In [59]:
s

array([24576, 24576, 24576, ..., 24576, 24576, 24576],
      shape=(96000000,), dtype=uint16)

In [57]:
test = s.compute()

ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (3,6521516)->(6521516,3) (3,6521516)->(6521516,3) (96000000,)->(96000000,newaxis) 

In [33]:
S = structure_tensor_3d(volume, 1, 3)

In [34]:
val, vec = eigenvalues(S)

In [35]:
test = val.compute()

ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (3,6080000)->(6080000,3) (3,6080000)->(6080000,3) (96000000,)->(96000000,newaxis) 